# Adaptive Attention Runner

Trains the adaptive-attention variant (Lu et al., 2017) of Show, Attend and Tell.
Dataset setup is identical to `drive_repo_one_click.ipynb` — only the training
and evaluation commands differ.

Manual setup before running:
1. Set Colab runtime to GPU.
2. Put this repo folder in `MyDrive/Neural_Image_Caption_Generator`.
3. Either put Flickr8k in `MyDrive/Neural_Image_Caption_Generator/data/flickr8k`, or set `FLICKR8K_SOURCE_DIR` below to wherever it is in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Change this only if your repo folder has a different name/location.
REPO_DIR = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')

# If Flickr8k is already inside the repo at data/flickr8k, leave this blank.
# If it is somewhere else in Drive, set it, e.g. Path('/content/drive/MyDrive/flickr8k').
FLICKR8K_SOURCE_DIR = None

assert REPO_DIR.exists(), f'Repo folder not found: {REPO_DIR}'
os.chdir(REPO_DIR)
print('Working in', Path.cwd())

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Prepare Flickr8k

If you do **not** have the dataset in Drive yet, upload your Kaggle `kaggle.json` in the next optional cell and then run the Kaggle download cell.

In [ ]:
import sys, os, csv, random as _random
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

data_root   = REPO_DIR / 'data' / 'flickr8k'
source_dir  = Path(FLICKR8K_SOURCE_DIR).expanduser().resolve() if FLICKR8K_SOURCE_DIR else (REPO_DIR / 'data' / 'flickr8k').resolve()
data_root.mkdir(parents=True, exist_ok=True)

# ── 1. images folder ────────────────────────────────────────────────────────
image_dir = None
for _c in source_dir.rglob('*'):
    if _c.is_dir() and _c.name.lower() in {'images', 'flickr8k_dataset', 'flicker8k_dataset'}:
        if len(list(_c.glob('*.jpg'))) > 100:
            image_dir = _c
            break
assert image_dir, f'No images folder found under {source_dir}'
img_dst = data_root / 'images'
if not img_dst.exists() and not img_dst.is_symlink():
    os.symlink(image_dir.resolve(), img_dst)
    print(f'Linked images folder: {image_dir}')

# ── 2. captions.txt ─────────────────────────────────────────────────────────
caps_src = next(source_dir.rglob('captions.txt'), None)
assert caps_src, f'captions.txt not found under {source_dir}'
caps_dst = data_root / 'captions.txt'
if not caps_dst.exists() and not caps_dst.is_symlink():
    os.symlink(caps_src.resolve(), caps_dst)
    print('Linked captions.txt')

# ── 3. Split files – link if present, otherwise auto-generate ───────────────
SPLIT_NAMES = ['Flickr_8k.trainImages.txt', 'Flickr_8k.devImages.txt', 'Flickr_8k.testImages.txt']
for _fname in SPLIT_NAMES:
    _src = next(source_dir.rglob(_fname), None)
    if _src:
        _dst = data_root / _fname
        if not _dst.exists() and not _dst.is_symlink():
            os.symlink(_src.resolve(), _dst)

_splits_generated = False
if any(not (data_root / f).exists() for f in SPLIT_NAMES):
    print('Split files missing – generating from captions.txt ...')
    _names = set()
    with open(caps_dst, newline='') as _f:
        for _raw in _f:
            _line = _raw.strip()
            if not _line: continue
            if '\t' in _line:
                _img_id = _line.split('\t', 1)[0]
            else:
                _row = next(csv.reader([_line]))
                if not _row or _row[0].lower() in {'image', 'image_name', 'filename'}: continue
                _img_id = _row[0]
            _names.add(os.path.basename(_img_id.strip().split('#')[0]))
    _names = sorted(_names)
    _rng = _random.Random(42)
    _rng.shuffle(_names)
    _n = len(_names)
    print(f'  {_n} unique images found')
    # Fixed val/test pools; ALL remaining images go to training.
    _n_val   = min(1000, max(1, _n // 8))
    _n_test  = min(1000, max(1, _n // 8))
    _n_train = _n - _n_val - _n_test
    for _fname, _imgs in [
        ('Flickr_8k.trainImages.txt', _names[:_n_train]),
        ('Flickr_8k.devImages.txt',   _names[_n_train:_n_train+_n_val]),
        ('Flickr_8k.testImages.txt',  _names[_n_train+_n_val:_n_train+_n_val+_n_test]),
    ]:
        _dst = data_root / _fname
        if not _dst.exists():
            _dst.write_text('\n'.join(_imgs) + '\n')
            print(f'  Generated {_fname} ({len(_imgs)} images)')
    _splits_generated = True

# ── 4. Validate (skip strict count check when we generated splits) ───────────
from utils import validate_dataset_layout
validate_dataset_layout(str(data_root), strict_split_counts=not _splits_generated)
print('Dataset ready:', data_root)

### Optional Kaggle Download Path

Only use these two cells if Flickr8k is not already in Drive. In Kaggle, create an API token and upload `kaggle.json` here.

In [ ]:
# OPTIONAL: upload kaggle.json, then run the next cell.
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPTIONAL: Kaggle download. Uncomment only if you used the kaggle.json cell above.
# !python scripts/colab_setup.py --repo_dir "$REPO_DIR" --download_kaggle

## Validate Code

In [ ]:
!pytest -q

## Train (Adaptive Attention)

Checkpoints are saved to `checkpoints/adaptive_epoch_NNN.pt`.
Best checkpoint (by val BLEU-4) is saved to `checkpoints/adaptive_best.pt`.
Training log (loss, mean sentinel weight β, BLEU scores per epoch) is written to
`results/training_log_adaptive.csv`.

In [ ]:
!python train_adaptive.py

## Evaluate

In [ ]:
# evaluate.py auto-detects the adaptive checkpoint via the saved model_type field
!python evaluate.py \
  --checkpoint checkpoints/adaptive_best.pt \
  --data_root data/flickr8k \
  --vocab data/flickr8k/vocab.json \
  --split test \
  --batch_size 64 \
  --results_out results/test_bleu_adaptive.json

## Visualize Attention + Sentinel Weights

Each word subplot shows `word` and `β=0.xx` — the sentinel weight at that step.
High β means the model relied on language statistics rather than the image.

In [ ]:
# visualize.py also auto-detects the adaptive checkpoint
!python visualize.py \
  --checkpoint checkpoints/adaptive_best.pt \
  --vocab data/flickr8k/vocab.json \
  --data_root data/flickr8k \
  --split test \
  --num_examples 6 \
  --output_dir results/attention_examples_adaptive \
  --overlay_style paper \
  --dpi 200

In [ ]:
from IPython.display import Image, display
for path in sorted(Path('results/attention_examples_adaptive').glob('*.png'))[:6]:
    print(path)
    display(Image(filename=str(path)))

## Analyse Sentinel Weights Over Training

Plots BLEU-4 and mean β per epoch so you can see how the model balances
visual attention vs. language-model reliance as training progresses.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('results/training_log_adaptive.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(log['epoch'], log['val_bleu4'] * 100, marker='o', label='Val BLEU-4')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BLEU-4 (%)')
axes[0].set_title('Validation BLEU-4')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(log['epoch'], log['mean_beta'], marker='o', color='orange', label='Mean β')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Sentinel Weight β')
axes[1].set_title('Sentinel Weight Over Training\n(high β = relying on language model)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('results/adaptive_training_curves.png', dpi=150)
plt.show()
print('Saved to results/adaptive_training_curves.png')